# Web Traffic: "Overview" & Stats

Dashboard exported from example-mcp-dashbuilder


In [ ]:
from elasticsearch import Elasticsearch
import pandas as pd
import matplotlib.pyplot as plt

es = Elasticsearch(
    "http://localhost:9200",
    basic_auth=("elastic", "changeme"),
)


## Top 5 response codes over time

Chart type: **bar**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs
    | STATS count = COUNT(*) BY time_bucket = BUCKET(@timestamp, 1 day), response_code = TO_STRING(response)
    | SORT time_bucket
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
df.plot.bar(x="time_bucket", y=["count"], title="Top 5 response codes over time")
plt.tight_layout()
plt.show()


## Bytes over time (hourly)

Chart type: **line**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs
    | STATS total_bytes = SUM(bytes) BY hour_bucket = BUCKET(@timestamp, 1 hour)
    | SORT hour_bucket
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
df.plot.line(x="hour_bucket", y=["total_bytes"], title="Bytes over time (hourly)")
plt.tight_layout()
plt.show()


## Top countries by request count

Chart type: **pie**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs
    | STATS requests = COUNT(*) BY country = geo.dest
    | SORT requests DESC
    | LIMIT 6
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
df.set_index("country")[["requests"]].plot.pie(subplots=True, title="Top countries by request count")
plt.tight_layout()
plt.show()


## Total requests

Chart type: **metric**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | WHERE @timestamp >= NOW() - 7 days AND @timestamp <= NOW() | STATS total_requests = COUNT(*)
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
from IPython.display import HTML, display
value = df.iloc[0]['total_requests']
display(HTML(f'''
<div style="padding: 20px 24px; border: 1px solid #e0e0e0; border-radius: 8px; background: #fafafa; display: inline-block; min-width: 220px;">
  <div style="font-size: 13px; color: #666; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 8px;">Total requests</div>
  <div style="font-size: 42px; font-weight: 600; color: #1a1a1a; line-height: 1;">{value}</div>
</div>
'''))


In [ ]:
# Total requests — trend sparkline
trend_result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | WHERE @timestamp >= NOW() - 7 days AND @timestamp <= NOW() | STATS total_requests = COUNT(*) BY day_bucket = BUCKET(@timestamp, 1 day) | SORT day_bucket
    """,
    format="json"
)

trend_df = pd.DataFrame(trend_result["values"], columns=[c["name"] for c in trend_result["columns"]])
trend_df.plot(title="Total requests — Trend")
plt.tight_layout()
plt.show()


## Requests by hour and day of week

Chart type: **heatmap**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs
    | EVAL day = DATE_FORMAT("EEEE", @timestamp), hour = DATE_FORMAT("HH", @timestamp)
    | STATS requests = COUNT(*) BY day, hour
    | SORT day, hour
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
pivot = df.pivot_table(index="day", columns="hour", values="requests")
fig, ax = plt.subplots()
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.values[i, j]:.0f}", ha="center", va="center")
fig.colorbar(im, ax=ax)
ax.set_title("Requests by hour and day of week")
plt.tight_layout()
plt.show()
